# DDEA Proteomics Course — Pre-Session Setup

**Run this notebook ONCE before the session (at home or the day before).**
It installs all R packages to your Google Drive so the main session notebook starts in seconds.

### Steps
1. Make sure the runtime is set to **R**: Runtime → Change runtime type → R.
2. **Mount Google Drive first**: click the folder icon in the **left sidebar**, then click the Google Drive icon. Authorise when prompted.
3. Run **Cell 1** — it checks Drive is mounted and sets up the library folder.
4. Run **Cell 2** — installs all packages (takes a few minutes; you only ever need to do this once).

> **Note:** The packages are compiled for the Linux environment that Colab uses. Do not copy this library folder to your own computer — it won't work there.

In [ ]:
# === Cell 1: Mount Google Drive ===
# This will prompt for authorisation if not already connected.
# Complete the prompt in the output below, then proceed to Cell 2.

system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\"")

# Poll until the filesystem is accessible
for (i in seq_len(15)) {
  if (dir.exists('/content/drive/MyDrive')) break
  Sys.sleep(2)
}

if (!dir.exists('/content/drive/MyDrive')) {
  cat('Drive did not mount. Complete the authorisation prompt above and re-run this cell.\n')
  stop('Drive not mounted.', call. = FALSE)
}

LIB_DIR <- '/content/drive/MyDrive/DDEA_proteomics_R_libs'
dir.create(LIB_DIR, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(LIB_DIR, .libPaths()))

cat('Drive mounted. Library folder:', LIB_DIR, '\n')
cat('Packages already installed:', length(list.dirs(LIB_DIR, recursive = FALSE)), '\n')

In [ ]:
# === Cell 2: Install all course packages to Drive ===
# Uses pak for parallel downloads + Posit Package Manager pre-compiled binaries.
# Safe to re-run — only installs packages not already present in LIB_DIR.

LIB_DIR <- '/content/drive/MyDrive/DDEA_proteomics_R_libs'
.libPaths(c(LIB_DIR, .libPaths()))

# Pre-compiled binaries for Ubuntu 22.04 (what Colab runs)
options(repos = c(PPM = 'https://packagemanager.posit.co/cran/__linux__/jammy/latest'))

# Bootstrap pak itself into LIB_DIR — it orchestrates everything below
if (!requireNamespace('pak', quietly = TRUE)) {
  install.packages('pak', lib = LIB_DIR)
}

# Package list — pak understands three formats:
#   plain name   → CRAN
#   bioc::name   → Bioconductor
#   user/repo    → GitHub
all_pkgs <- c(
  'dplyr', 'forcats', 'ggplot2', 'ggplotify', 'pheatmap', 'plotly',
  'purrr', 'scales', 'stringr', 'tibble', 'tidyr', 'tidyverse', 'Seurat',
  'bioc::BiocParallel', 'bioc::clusterProfiler', 'bioc::limma',
  'bioc::miloR', 'bioc::org.Hs.eg.db', 'bioc::PhosR', 'bioc::SingleCellExperiment',
  'aj-grant/navmix'
)

# Strip spec prefix to get the bare package name for the already-installed check
pkg_base <- function(spec) sub('^.+/', '', sub('^bioc::', '', spec))

installed_pkgs <- rownames(installed.packages(lib.loc = LIB_DIR))
new_pkgs <- all_pkgs[!sapply(all_pkgs, function(p) pkg_base(p) %in% installed_pkgs)]

if (length(new_pkgs) == 0) {
  cat('All packages already installed.\n')
} else {
  cat('Installing', length(new_pkgs), 'package(s) in parallel via pak:\n')
  cat(paste(' -', new_pkgs, collapse = '\n'), '\n\n')
  pak::pak(new_pkgs, lib = LIB_DIR, ask = FALSE)
}

cat('\n=== Done! ===\n')
cat('Packages saved to', LIB_DIR, '\n')
cat('Total packages installed:', length(list.dirs(LIB_DIR, recursive = FALSE)), '\n')
cat('You can now close this notebook.\n')
cat('During the session, the main notebook will load from Drive automatically.\n')

### What to do if the install fails midway

Just re-run Cell 2 — it skips packages that are already installed and only retries the ones that failed.

### What to do if Colab updates its R version between now and the session

Re-run both cells. The library folder will be updated with binaries matching the new R version. This is rare but possible if Google updates the Colab runtime between when you pre-install and when the session runs.